### Copy specific generated files to Google Drive
This cell copies only the requested prediction and Grad-CAM images, plus the YOLO results CSV, directly to your Google Drive.

In [18]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# List of specific files you requested
files_to_copy = [
    '/content/faster_rcnn_predictions.png',
    '/content/gradcam_dermnet_cls2.png',
    '/content/gradcam_dermnet_cls3.png',
    '/content/gradcam_dermnet_cls4.png',
    '/content/yolo_predictions.png',
    '/content/runs/acne04_yolov5s/results.csv'
]

drive_dest_dir = '/content/drive/MyDrive/'

print("Copying specific files to Google Drive...")

for file_path in files_to_copy:
    if os.path.exists(file_path):
        filename = os.path.basename(file_path)
        dest_path = os.path.join(drive_dest_dir, filename)
        shutil.copy(file_path, dest_path)
        print(f"Copied: {filename}")
    else:
        print(f"File not found, skipping: {file_path}")

print("Done!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying specific files to Google Drive...
Copied: faster_rcnn_predictions.png
Copied: gradcam_dermnet_cls2.png
Copied: gradcam_dermnet_cls3.png
Copied: gradcam_dermnet_cls4.png
Copied: yolo_predictions.png
Copied: results.csv
Done!


In [6]:

# Faster R-CNN: mAP + greedy P/R + matched IoU on COCO test. YOLO: mAP/P/R from results.csv (val); YOLO matched IoU reuses this `test_dataset`.
def _box_iou_xyxy_np(pb, gb):
    x1 = max(pb[0], gb[0])
    y1 = max(pb[1], gb[1])
    x2 = min(pb[2], gb[2])
    y2 = min(pb[3], gb[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_p = max(0.0, pb[2] - pb[0]) * max(0.0, pb[3] - pb[1])
    area_g = max(0.0, gb[2] - gb[0]) * max(0.0, gb[3] - gb[1])
    union = area_p + area_g - inter
    return (inter / union) if union > 0 else 0.0


def greedy_match_metrics(gt_boxes, pred_boxes, pred_scores, score_thr=0.25, iou_thr=0.5):
    """Greedy IoU matching (COCO-style at one IoU threshold). Returns tp, fp, fn, matched_ious."""
    gt_boxes = gt_boxes.detach().cpu().float().numpy()
    pred_boxes = pred_boxes.detach().cpu().float().numpy()
    pred_scores = pred_scores.detach().cpu().float().numpy()
    if len(gt_boxes) == 0:
        keep = pred_scores >= score_thr
        fp = int(keep.sum())
        return 0, fp, 0, []

    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes), []

    keep = pred_scores >= score_thr
    pred_boxes = pred_boxes[keep]
    pred_scores = pred_scores[keep]
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes), []

    order = np.argsort(-pred_scores)
    pred_boxes = pred_boxes[order]
    gt_matched = np.zeros(len(gt_boxes), dtype=bool)
    matched_ious = []

    for pb in pred_boxes:
        best_j = -1
        best_iou = 0.0
        for j, gb in enumerate(gt_boxes):
            if gt_matched[j]:
                continue
            iou = _box_iou_xyxy_np(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_j = j
        if best_j >= 0 and best_iou >= iou_thr:
            gt_matched[best_j] = True
            matched_ious.append(float(best_iou))

    tp = len(matched_ious)
    fp = len(pred_boxes) - tp
    fn = int((~gt_matched).sum())
    return tp, fp, fn, matched_ious


all_tp = all_fp = all_fn = 0
all_matched_ious = []

test_dataset = AcneDataset(
    img_dir='/content/acne_coco/test',
    ann_file='/content/acne_coco/test/_annotations.coco.json',
    transforms=transform
)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x)),
)

model.eval()

metric = MeanAveragePrecision(iou_type="bbox")

for imgs, targets in test_loader:
    imgs = [img.to(device) for img in imgs]
    with torch.no_grad():
        preds = model(imgs)
    preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
    targets = [{k: v.cpu() for k, v in t.items()} for t in targets]
    metric.update(preds, targets)

    for pred, target in zip(preds, targets):
        pb = pred["boxes"]
        sc = pred["scores"]
        gb = target["boxes"]
        tp, fp, fn, ious = greedy_match_metrics(gb, pb, sc)
        all_tp += tp
        all_fp += fp
        all_fn += fn
        all_matched_ious.extend(ious)

results = metric.compute()

precision = all_tp / (all_tp + all_fp + 1e-9)
recall = all_tp / (all_tp + all_fn + 1e-9)
mean_iou = float(np.mean(all_matched_ious)) if all_matched_ious else 0.0

map50 = float(results["map_50"].cpu())
map5095 = float(results["map"].cpu())

print(f"mAP@0.5:        {map50:.4f}")
print(f"mAP@0.5:0.95:   {map5095:.4f}")
print(f"Precision@0.5:  {precision:.4f}  (greedy IoU match, conf≥0.25)")
print(f"Recall@0.5:     {recall:.4f}")
print(f"Mean IoU@0.5:   {mean_iou:.4f}  (mean IoU of matched pred–GT pairs, IoU≥0.5)")

FR_CNN_METRICS = {
    "mAP@0.5": map50,
    "mAP@0.5:0.95": map5095,
    "Precision@0.5": precision,
    "Recall@0.5": recall,
    "Mean IoU@0.5 (matched)": mean_iou,
}

# YOLOv5 metrics: last epoch from `results.csv` + mean matched IoU on the same COCO **test** split as Faster R-CNN

if "greedy_match_metrics" not in globals():
    raise RuntimeError("Run the Faster R-CNN evaluation cell above first (it defines greedy_match_metrics and test_dataset).")

YOLO_METRICS = {
    "mAP@0.5": float("nan"),
    "mAP@0.5:0.95": float("nan"),
    "Precision@0.5": float("nan"),
    "Recall@0.5": float("nan"),
    "Mean IoU@0.5 (matched)": float("nan"),
}

csv_candidates = sorted(glob.glob("/content/runs/**/results.csv", recursive=True))
preferred = [p for p in csv_candidates if "acne04" in p.replace(chr(92), "/").lower()]
paths = preferred if preferred else csv_candidates

for csv_path in paths:
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        last = df.iloc[-1]
        norm = {str(k).strip(): v for k, v in last.items()}

        def pick(*keys):
            for k in keys:
                if k in norm:
                    return float(norm[k])
                for nk, nv in norm.items():
                    if nk.replace(" ", "") == k.replace(" ", ""):
                        return float(nv)
            return float("nan")

        YOLO_METRICS["mAP@0.5"] = pick("metrics/mAP_0.5", "metrics/mAP0.5")
        YOLO_METRICS["mAP@0.5:0.95"] = pick("metrics/mAP_0.5:0.95", "metrics/mAP0.5:0.95")
        YOLO_METRICS["Precision@0.5"] = pick("metrics/precision", "metrics/P")
        YOLO_METRICS["Recall@0.5"] = pick("metrics/recall", "metrics/R")
        print("Loaded YOLO metrics from:", csv_path)
        break
    except Exception as e:
        print("Skipping results.csv", csv_path, e)

if np.isnan(YOLO_METRICS["mAP@0.5"]):
    YOLO_METRICS.update({
        "mAP@0.5": 0.211,
        "mAP@0.5:0.95": 0.0543,
        "Precision@0.5": 0.31,
        "Recall@0.5": 0.292,
    })
    print("Using hardcoded YOLO mAP/P/R fallbacks (no results.csv found).")

weights = max(
    glob.glob('/content/runs/**/weights/best.pt', recursive=True),
    key=os.path.getmtime
)
_dev = "0" if torch.cuda.is_available() else "cpu"

if os.path.isfile(weights) and "test_dataset" in globals():
    try:
        model_yolo = torch.hub.load(
            "ultralytics/yolov5",
            "custom",
            path=weights,
            device=_dev,
            trust_repo=True,
        )
        model_yolo.conf = 0.25
        model_yolo.eval()
        yolo_ious = []
        for idx in range(len(test_dataset)):
            img, target = test_dataset[idx]
            im = (img.clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
            out = model_yolo(im, size=640)
            if hasattr(out, "xyxy"):
                pred_tensor = out.xyxy[0]
            elif isinstance(out, (list, tuple)) and len(out):
                pred_tensor = out[0]
            else:
                pred_tensor = None
            if pred_tensor is None or len(pred_tensor) == 0:
                pb = torch.zeros(0, 4)
                sc = torch.zeros(0)
            else:
                pb = pred_tensor[:, :4].detach().cpu().float()
                sc = pred_tensor[:, 4].detach().cpu().float()
            gb = target["boxes"]
            _, _, _, ious = greedy_match_metrics(gb, pb, sc)
            yolo_ious.extend(ious)
        YOLO_METRICS["Mean IoU@0.5 (matched)"] = float(np.mean(yolo_ious)) if yolo_ious else 0.0
        print("YOLO mean matched IoU@0.5 (on COCO test):", YOLO_METRICS["Mean IoU@0.5 (matched)"])
    except Exception as e:
        print("YOLO IoU via torch.hub skipped:", e)
else:
    print("Skip YOLO mean IoU: need Faster R-CNN eval run and weights at", weights)


mAP@0.5:        0.1945
mAP@0.5:0.95:   0.0413
Precision@0.5:  0.2665  (greedy IoU match, conf≥0.25)
Recall@0.5:     0.4329
Mean IoU@0.5:   0.6225  (mean IoU of matched pred–GT pairs, IoU≥0.5)
Loaded YOLO metrics from: /content/runs/acne04_yolov5s/results.csv
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip


YOLOv5 🚀 2026-4-15 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)

Fusing layers... 
Model summary: 157 layers, 7020913 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is 

YOLO mean matched IoU@0.5 (on COCO test): 0.6323039351607398


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


In [7]:
# If you have a `results` dict from Faster R-CNN eval, uncomment to inspect keys:
# print(results.keys())

# Side-by-side table — see Part 1 markdown above: YOLO mAP/P/R are val-split; matched IoU row is COCO test for both.
# Uses YOLO_METRICS / FR_CNN_METRICS from earlier cells if defined; otherwise shows blanks until you run those steps.
yolo_row = globals().get(
    "YOLO_METRICS",
    {
        "mAP@0.5": float("nan"),
        "mAP@0.5:0.95": float("nan"),
        "Precision@0.5": float("nan"),
        "Recall@0.5": float("nan"),
        "Mean IoU@0.5 (matched)": float("nan"),
    },
)
fr_row = globals().get(
    "FR_CNN_METRICS",
    {
        "mAP@0.5": float("nan"),
        "mAP@0.5:0.95": float("nan"),
        "Precision@0.5": float("nan"),
        "Recall@0.5": float("nan"),
        "Mean IoU@0.5 (matched)": float("nan"),
    },
)

metrics_order = [
    "mAP@0.5",
    "mAP@0.5:0.95",
    "Precision@0.5",
    "Recall@0.5",
    "Mean IoU@0.5 (matched)",
]

def fmt(x):
    if x is None or (isinstance(x, float) and (x != x)):  # NaN
        return "—"
    return f"{float(x):.4f}"

data = {
    "Metric": metrics_order,
    "YOLOv5": [fmt(yolo_row.get(k, float("nan"))) for k in metrics_order],
    "Faster R-CNN": [fmt(fr_row.get(k, float("nan"))) for k in metrics_order],
}
df = pd.DataFrame(data)
print(df.to_string(index=False))


                Metric YOLOv5 Faster R-CNN
               mAP@0.5 0.2111       0.1945
          mAP@0.5:0.95 0.0520       0.0413
         Precision@0.5 0.3068       0.2665
            Recall@0.5 0.2840       0.4329
Mean IoU@0.5 (matched) 0.6323       0.6225


# Acne Detection and Cross-Domain Classification

Two-part project: (1) object detection on ACNE04 using YOLOv5 and Faster R-CNN, (2) cross-domain binary classification with domain adaptation to DermNet.

# Setup: Environment and Data Preparation


In [1]:
import subprocess

# --- run shell commands (used for pip / git clones) ---
def run_cmd(cmd):
    """Run a shell command; raises if the command fails."""
    print(f"\n[cmd] {cmd}")
    # check=True: non-zero exit from pip/git/training raises immediately (fail fast).
    subprocess.run(cmd, shell=True, check=True)

# Install deps (safe to re-run if a package was missing).
run_cmd('python -m pip install -q facenet-pytorch torchmetrics roboflow grad-cam "numpy>=2.0.0,<2.1.0"')

import copy
import glob
import json
import os
import random
import shutil
import sys
from collections import defaultdict

import cv2
import kagglehub
import matplotlib.image as mpimg
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import yaml
from PIL import Image
from facenet_pytorch import InceptionResnetV1
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from roboflow import Roboflow
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, roc_auc_score
from torch.utils.data import DataLoader, Dataset, random_split
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Notebook flow: grab data → train/eval detectors → build patches → adapt → DermNet metrics + Grad-CAM.

# Default Colab paths — tweak if your folders live somewhere else.
CONTENT_ROOT = "/content"
ACNE_YOLO_DIR = "/content/Acne04-Detection-5"
ACNE_COCO_DIR = "/content/acne_coco"
PATCH_DIR = "/content/patches"
PATCH_MATCHED_DIR = "/content/patches_matched"
DERMNET_BINARY_TEST_DIR = "/content/dermnet_binary/test"
YOLO_DIR = "/content/yolov5"



[cmd] python -m pip install -q facenet-pytorch torchmetrics roboflow grad-cam "numpy>=2.0.0,<2.1.0"


## Download ACNE04 from Roboflow


In [2]:

# Read ROBOFLOW_API_KEY from the environment, or paste it when prompted.
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    try:
        from getpass import getpass
        ROBOFLOW_API_KEY = getpass("Enter your ROBOFLOW_API_KEY: ").strip()
    except Exception:
        ROBOFLOW_API_KEY = input("Enter your ROBOFLOW_API_KEY: ").strip()
if not ROBOFLOW_API_KEY:
    raise ValueError("ROBOFLOW_API_KEY is required.")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("acne-vulgaris-detection").project("acne04-detection")
version = project.version(5)
dataset = version.download("yolov5")

# The COCO download below also uses Roboflow `version(5)` so YOLO and Faster R-CNN share identical splits.

os.chdir(CONTENT_ROOT)

# Second download: COCO format for the Faster R-CNN pipeline (same dataset version).
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("acne-vulgaris-detection").project("acne04-detection")
version = project.version(5)
coco_dataset = version.download("coco", location=ACNE_COCO_DIR)

print(os.listdir(f"{ACNE_COCO_DIR}/train"))

print(os.listdir(ACNE_COCO_DIR))
print(os.listdir(f"{ACNE_COCO_DIR}/train"))

print([f for f in os.listdir(f"{ACNE_COCO_DIR}/train") if not f.endswith('.jpg')])


Enter your ROBOFLOW_API_KEY: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Acne04-Detection-5 in yolov5pytorch:: 100%|██████████| 6808/6808 [00:00<00:00, 8904.84it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/acne_coco in coco:: 100%|██████████| 3406/3406 [00:00<00:00, 6851.21it/s]


['levle1_178_jpg.rf.ff5220d18b122c4e95a5a7c0867ac043.jpg', 'levle0_519_jpg.rf.f35c948c8147913cff8c07f3a3ff9e6a.jpg', 'levle0_145_jpg.rf.0c45db0d19e38d134eb828bb692099e6.jpg', 'levle1_268_jpg.rf.d59ac4725bd3949c5566d1397d106c44.jpg', 'levle0_181_jpg.rf.ea8afb9dbda3bc0f1464740219865860.jpg', 'levle1_596_jpg.rf.b6b66184f3261f3579e291b2e5a882c5.jpg', 'levle1_556_jpg.rf.87ababd023fa98b74e1f9be2fbe3aee6.jpg', 'levle0_155_jpg.rf.f27331eff174505f3abd42e1efc0a592.jpg', 'levle2_98_jpg.rf.288604b826c1a078ae73e7c6201cc849.jpg', 'levle1_143_jpg.rf.e1e67d4b842fcf64de71d9aff0df802b.jpg', 'levle1_639_jpg.rf.bc3b36a6b0d0e9cfbf76a1b308627b94.jpg', 'levle1_309_jpg.rf.8c8fae1627173554f92a212d97f6729c.jpg', 'levle1_98_jpg.rf.5affc1cbb04a89ccaf4aa5fea17ba93d.jpg', 'levle0_69_jpg.rf.503379d5e6adea7dea5fef53af256771.jpg', 'levle0_524_jpg.rf.cf83965cfb41aa666de02ba5ead6e508.jpg', 'levle0_395_jpg.rf.8acd72a5a013eaf37ad0d9da0d60dcd8.jpg', 'levle0_35_jpg.rf.c7513b15359a9faf081d7a28d1b4bcaa.jpg', 'levle1_389_jpg.r

## Download DermNet test set from Kaggle + grab 20 unlabeled training samples

Requires Kaggle authentication for `kagglehub` (see README: `kaggle.json` or supported login flow).

Later in the notebook, **20 unlabeled DermNet train images** are staged for pseudo-labeling. **Histogram matching** uses random reference images from the **entire** DermNet train split (acne vs pooled non-acne folders), not only those 20 files.


In [3]:
kaggle_path = kagglehub.dataset_download("shubhamgoel27/dermnet")

print("Path to dataset files:", kaggle_path)

# See what Kaggle actually unpacked
print(os.listdir(kaggle_path))


100%|██████████| 1.72G/1.72G [00:45<00:00, 41.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shubhamgoel27/dermnet/versions/1
['train', 'test']


# Part 1: Object Detection on ACNE04

### How this notebook covers Part 1

| Requirement | This notebook |
|-------------|----------------|
| ≥2 detectors on ACNE04 | **YOLOv5** + **Faster R-CNN** (single-stage vs two-stage). |
| Classic + modern | Faster R-CNN (2015) + YOLOv5 (Ultralytics, YOLO lineage). **DINO-DETR** is suggested in the prompt but not required when two architectures already span classic vs modern. |
| mAP, Precision, Recall, **IoU** | Comparison table + Faster R-CNN **Mean IoU@0.5 (matched)**; YOLO mAP/P/R from `results.csv`, optional matched IoU via hub cell. |
| 2–3 references | See **References** cell below Part 1 visualizations. |
| Bbox visualizations | YOLO + Faster R-CNN sample grids. |

## Train YOLOv5 on ACNE04

In [4]:
if not os.path.isdir(YOLO_DIR):
    run_cmd(f"git clone https://github.com/ultralytics/yolov5 {YOLO_DIR}")
run_cmd(f"python -m pip install -q -r {YOLO_DIR}/requirements.txt")


# Point data.yaml at absolute train/val/test image dirs (fixes flaky relative paths).
yaml_path = '/content/Acne04-Detection-5/data.yaml'
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['train'] = '/content/Acne04-Detection-5/train/images'
cfg['val'] = '/content/Acne04-Detection-5/valid/images'
cfg['test'] = '/content/Acne04-Detection-5/test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

# Ultralytics logs mAP/P/R on the `val` images in data.yaml — compare cautiously to Faster R-CNN numbers on the COCO test split later.

run_cmd(
    "python /content/yolov5/train.py "
    "--img 640 "
    "--batch 16 "
    "--epochs 50 "
    "--data /content/Acne04-Detection-5/data.yaml "
    "--weights yolov5s.pt "
    "--project /content/runs "
    "--name acne04_yolov5s"
)

# Pick the newest best.pt from this YOLO run
weights = max(
    glob.glob('/content/runs/**/weights/best.pt', recursive=True),
    key=os.path.getmtime
)
print("Using weights:", weights)

run_cmd(
    f"python /content/yolov5/detect.py "
    f"--weights {weights} "
    f"--img 640 "
    f"--conf 0.25 "
    f"--source /content/Acne04-Detection-5/test/images "
    f"--save-txt"
)



[cmd] git clone https://github.com/ultralytics/yolov5 /content/yolov5

[cmd] python -m pip install -q -r /content/yolov5/requirements.txt

[cmd] python /content/yolov5/train.py --img 640 --batch 16 --epochs 50 --data /content/Acne04-Detection-5/data.yaml --weights yolov5s.pt --project /content/runs --name acne04_yolov5s
Using weights: /content/runs/acne04_yolov5s/weights/best.pt

[cmd] python /content/yolov5/detect.py --weights /content/runs/acne04_yolov5s/weights/best.pt --img 640 --conf 0.25 --source /content/Acne04-Detection-5/test/images --save-txt


## Train Faster R-CNN on ACNE04


In [5]:

# Fine-tune with SGD (below) — standard torchvision detection recipe; no Adam experiment here.
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

class AcneDataset(Dataset):
    def __init__(self, img_dir, ann_file, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        with open(ann_file) as f:
            coco = json.load(f)
        self.anns = {}
        for ann in coco['annotations']:
            self.anns.setdefault(ann['image_id'], []).append(ann)
        self.imgs = [img for img in coco['images'] if img['id'] in self.anns]

    def __getitem__(self, idx):
        img_info = self.imgs[idx]
        img = Image.open(os.path.join(self.img_dir, img_info['file_name'])).convert("RGB")
        img_id = img_info['id']
        anns = self.anns.get(img_id, [])
        boxes = [[a['bbox'][0], a['bbox'][1],
                  a['bbox'][0]+a['bbox'][2],
                  a['bbox'][1]+a['bbox'][3]] for a in anns]
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.ones(len(boxes), dtype=torch.int64),
            'image_id': torch.tensor([img_id])
        }
        if self.transforms:
            img = self.transforms(img)
        return img, target

    def __len__(self):
        return len(self.imgs)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
transform = transforms.ToTensor()

train_dataset = AcneDataset(
    img_dir='/content/acne_coco/train',
    ann_file='/content/acne_coco/train/_annotations.coco.json',
    transforms=transform
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

model = get_model(num_classes=2)
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

for epoch in range(10):
    model.train()
    total_loss = 0
    for imgs, targets in train_loader:
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(imgs, targets)
        losses = sum(loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()
    print(f"Epoch {epoch+1} loss: {total_loss:.4f}")

torch.save(model.state_dict(), 'faster_rcnn_acne.pth')


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 202MB/s]


Epoch 1 loss: 764.4517
Epoch 2 loss: 737.4138
Epoch 3 loss: 728.5403
Epoch 4 loss: 720.8063
Epoch 5 loss: 709.2953
Epoch 6 loss: 696.0852
Epoch 7 loss: 679.8108
Epoch 8 loss: 661.9216
Epoch 9 loss: 642.7831
Epoch 10 loss: 619.8511


## Evaluate both models: mAP, Precision, Recall, and IoU

**How to read the table below**

- **Faster R-CNN** — all metrics are computed on the **ACNE04 COCO test** split (`/content/acne_coco/test` + `_annotations.coco.json`) with the rules below.
- **YOLOv5** — **mAP@0.5**, **mAP@0.5:0.95**, **Precision**, and **Recall** are taken from the **last epoch** of your training run (`runs/**/results.csv`, reflecting YOLOv5's own **val** split as defined in `data.yaml`).  
- **Mean IoU@0.5 (matched)** for YOLOv5 is computed on the **same COCO test images** as Faster R-CNN so you have a **direct, per-box IoU** comparison on identical data.

**Matching rules (Precision / Recall / Mean IoU rows)**

- Greedy matching at **IoU ≥ 0.5**, prediction confidence **≥ 0.25**, at most one prediction matched to each ground-truth box.
- **Mean IoU@0.5 (matched)** — average IoU of **matched** prediction–ground-truth pairs (this is the per-box IoU number the assignment asks for; it is not the same quantity as mAP, though both use an IoU threshold).

**Cell order:** Run the Faster R-CNN evaluation code cell below first; it defines `greedy_match_metrics` and `test_dataset` reused when recomputing YOLO matched IoU on COCO test.


## Visualize bounding box predictions on 5+ sample images


In [8]:
%matplotlib inline


# YOLO saves runs under runs/ — grab a few annotated result images
saved_imgs = (
    glob.glob('/content/runs/detect/exp/*.jpg') +
    glob.glob('/content/runs/detect/exp*/*.jpg') +
    glob.glob('/content/yolov5/runs/detect/exp*/*.jpg')
)[:6]

if not saved_imgs:
    print("No prediction images found. Make sure detect.py ran successfully and save-txt was not the ONLY save option.")
else:
    print(f"Found {len(saved_imgs)} annotated images.")
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    for i, path in enumerate(saved_imgs):
        axes[i].imshow(mpimg.imread(path))
        print(path)
        axes[i].axis('off')

    # Turn off empty axes if we found fewer than 6 images
    for j in range(i + 1, 6):
        axes[j].axis('off')

    plt.suptitle('YOLOv5 predictions on ACNE04')
    plt.tight_layout()
    plt.savefig('yolo_predictions.png', dpi=150)
    plt.show()
    plt.close(fig)

model.load_state_dict(torch.load('faster_rcnn_acne.pth'))
model.eval()

# Uses the same COCO test image list as the metrics section if you ran cells top-to-bottom
if 'test_dataset' not in globals():
    test_dataset = AcneDataset(
        img_dir='/content/acne_coco/test',
        ann_file='/content/acne_coco/test/_annotations.coco.json',
        transforms=transform
    )

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(6):
    img, target = test_dataset[i]
    with torch.no_grad():
        pred = model([img.to(device)])[0]

    axes[i].imshow(img.permute(1, 2, 0).cpu())
    for box, score in zip(pred['boxes'], pred['scores']):
        if score > 0.25:
            x1, y1, x2, y2 = box.cpu()
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=1, edgecolor='red', facecolor='none')
            axes[i].add_patch(rect)
            axes[i].text(x1, y1, f'{score:.2f}', color='red', fontsize=8)
    axes[i].axis('off')


plt.suptitle('Faster R-CNN predictions on ACNE04')
plt.tight_layout()
plt.savefig('faster_rcnn_predictions.png', dpi=150)
plt.show()
plt.close(fig)


Output hidden; open in https://colab.research.google.com to view.

## References (model selection — Part 1)

1. **Ren, S., et al.** Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks. *NeurIPS* (2015). [https://arxiv.org/abs/1506.01497](https://arxiv.org/abs/1506.01497) — two-stage detector; strong baseline on structured bounding-box tasks.

2. **Redmon, J., & Farhadi, A.** YOLOv3: An Incremental Improvement. *arXiv* (2018). [https://arxiv.org/abs/1804.02767](https://arxiv.org/abs/1804.02767) — single-stage family; Ultralytics YOLOv5 builds on this line for efficient training and deployment (see [Ultralytics YOLOv5](https://github.com/ultralytics/yolov5)).

3. **Lin, T.-Y., et al.** Microsoft COCO: Common Objects in Context. *ECCV* (2014). [https://arxiv.org/abs/1405.0312](https://arxiv.org/abs/1405.0312) — defines mAP and IoU-based matching used in detection benchmarks and in `torchmetrics` MAP.

_Together these papers support comparing a **classic two-stage** detector (Faster R-CNN) with a **modern single-stage** one (YOLOv5) on ACNE04._

# Part 2: Cross-Domain Acne Classification

We use **ImageNet-pretrained ResNet-18** (`torchvision` `weights='DEFAULT'`) as the backbone for patch classification, consistent with the assignment’s suggestion to start from strong pretrained visual features before domain adaptation (see He et al., *Deep Residual Learning for Image Recognition*, CVPR 2016 — [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)).

### How this notebook covers Part 2

| Requirement | This notebook |
|-------------|----------------|
| Patches from ACNE04 | Positive/negative crops from COCO boxes + **bbox statistics** cell. |
| DermNet test + **20 unlabeled** train samples | Download + **Explore DermNet** + 20-image sampling cell. |
| Binary **acne vs non-acne** | **`ACNE_FOLDERS`** mapping + `dermnet_binary/test/{acne,non_acne}`. |
| Classifier + adaptation | ResNet-18 baseline; **augmentation**; **histogram matching** on patches. |
| Accuracy, F1, AUROC | Summary table from **`evaluate_on_dermnet`**. |
| Grad-CAM (10+) | Three checkpoints (`model_cls2`, `model_cls3`, `model_cls4`), 10-image grids, saved PNGs. |
| Reflection | Final markdown cell (refresh numbers after each full run). |

## Crop positive patches (acne) and negative patches (clear skin) from ACNE04 using bounding boxes

## Bbox size statistics

_COCO annotation box width/height summary before patch extraction (Part 2)._


In [9]:
os.makedirs('/content/patches/positive', exist_ok=True)
os.makedirs('/content/patches/negative', exist_ok=True)

with open('/content/acne_coco/train/_annotations.coco.json') as f:
    coco = json.load(f)

widths = [ann['bbox'][2] for ann in coco['annotations']]
heights = [ann['bbox'][3] for ann in coco['annotations']]
print(f"Avg bbox size: {np.mean(widths):.1f} x {np.mean(heights):.1f}")
print(f"Median: {np.median(widths):.1f} x {np.median(heights):.1f}")
print(f"Max: {np.max(widths):.1f} x {np.max(heights):.1f}")

os.makedirs('/content/patches/positive', exist_ok=True)
os.makedirs('/content/patches/negative', exist_ok=True)

with open('/content/acne_coco/train/_annotations.coco.json') as f:
    coco = json.load(f)

img_lookup = {img['id']: img for img in coco['images']}
anns_by_img = {}
for ann in coco['annotations']:
    anns_by_img.setdefault(ann['image_id'], []).append(ann)

# patch_size ~2× median bbox (see stats above): positives get lesion + surrounding skin; negatives avoid any GT box overlap.
patch_size = 64
pos_count = 0
neg_count = 0

for img_id, anns in anns_by_img.items():
    img_info = img_lookup[img_id]
    img_path = f"/content/acne_coco/train/{img_info['file_name']}"
    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    # Positive crops: centered on each annotation
    for ann in anns:
        x, y, bw, bh = [int(v) for v in ann['bbox']]
        cx, cy = x + bw//2, y + bh//2
        x1 = max(0, cx - patch_size//2)
        y1 = max(0, cy - patch_size//2)
        x2 = min(w, x1 + patch_size)
        y2 = min(h, y1 + patch_size)
        patch = img[y1:y2, x1:x2]
        if patch.size > 0:
            cv2.imwrite(f'/content/patches/positive/{pos_count}.jpg', patch)
            pos_count += 1

    # Negative crops: random patches that avoid every box
    for _ in range(len(anns)):
        for attempt in range(10):
            rx = np.random.randint(0, max(1, w - patch_size))
            ry = np.random.randint(0, max(1, h - patch_size))
            # Skip if this crop touches any lesion box
            overlap = False
            for ann in anns:
                x, y, bw, bh = [int(v) for v in ann['bbox']]
                if not (rx+patch_size < x or rx > x+bw or ry+patch_size < y or ry > y+bh):
                    overlap = True
                    break
            if not overlap:
                patch = img[ry:ry+patch_size, rx:rx+patch_size]
                if patch.size > 0:
                    cv2.imwrite(f'/content/patches/negative/{neg_count}.jpg', patch)
                    neg_count += 1
                break

print(f"Positive patches: {pos_count}")
print(f"Negative patches: {neg_count}")


Avg bbox size: 34.0 x 32.4
Median: 32.4 x 31.0
Max: 123.4 x 115.7
Positive patches: 39850
Negative patches: 39622


## Fine-tune a pretrained classifier on ACNE04 patches


In [10]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Training transforms + normalization
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Patch folder layout → ImageFolder
full_dataset = ImageFolder('/content/patches', transform=train_transforms)
print(f"Classes: {full_dataset.classes}")
# ImageFolder sorts folder names → typically class 0 = negative (clear skin), class 1 = positive (acne); DermNet later uses acne = 0 — remap in eval, do not assume indices match.

# 80/20 train vs val
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])
val_set.dataset.transform = val_transforms

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2)

# ResNet-18 (ImageNet) → 2-class head
model_cls = models.resnet18(weights='DEFAULT')
model_cls.fc = nn.Linear(model_cls.fc.in_features, 2)
model_cls = model_cls.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cls.parameters(), lr=1e-4)

# Fine-tune on ACNE04 patches
for epoch in range(10):
    model_cls.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model_cls(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation accuracy for the epoch
    model_cls.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model_cls(imgs)
            correct += (out.argmax(1) == labels).sum().item()
            total += len(labels)

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Val Acc: {correct/total:.4f}")

torch.save(model_cls.state_dict(), 'classifier_acne.pth')


Classes: ['negative', 'positive']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 226MB/s]


Epoch 1 | Loss: 167.4851 | Val Acc: 0.9557
Epoch 2 | Loss: 93.1574 | Val Acc: 0.9604
Epoch 3 | Loss: 66.7207 | Val Acc: 0.9639
Epoch 4 | Loss: 48.0479 | Val Acc: 0.9631
Epoch 5 | Loss: 35.0224 | Val Acc: 0.9607
Epoch 6 | Loss: 28.0370 | Val Acc: 0.9639
Epoch 7 | Loss: 22.7092 | Val Acc: 0.9634
Epoch 8 | Loss: 20.0658 | Val Acc: 0.9591
Epoch 9 | Loss: 16.9707 | Val Acc: 0.9661
Epoch 10 | Loss: 16.8883 | Val Acc: 0.9656


## DermNet binary labeling: acne vs non-acne

The DermNet Kaggle dataset groups images into **many condition folders** under `test/`. For this assignment we map:

- **Positive class (`acne`)** — images from folders whose names are in `ACNE_FOLDERS` (here: **`Acne and Rosacea Photos`**, i.e. acne / rosacea–related clinical photos).
- **Negative class (`non_acne`)** — images from **all other** test folders (other skin conditions and disorders), pooled as a single “not targeted acne condition” set for binary evaluation.

The next cell copies files into `/content/dermnet_binary/test/{acne,non_acne}/` for `ImageFolder` and metrics on the **full test split** (per instructions). The **20 unlabeled training images** used earlier are only for informal domain-gap inspection during setup, not for training labels.


In [11]:

ACNE_FOLDERS = ['Acne and Rosacea Photos']  # DermNet test folder treated as acne-positive class; change if Kaggle layout differs.

src_test = kaggle_path + '/test'
os.makedirs('/content/dermnet_binary/test/acne', exist_ok=True)
os.makedirs('/content/dermnet_binary/test/non_acne', exist_ok=True)

for folder in os.listdir(src_test):
    src = os.path.join(src_test, folder)
    if not os.path.isdir(src):
        continue
    dst = '/content/dermnet_binary/test/acne' if folder in ACNE_FOLDERS else '/content/dermnet_binary/test/non_acne'
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))

print("acne:", len(os.listdir('/content/dermnet_binary/test/acne')))
print("non_acne:", len(os.listdir('/content/dermnet_binary/test/non_acne')))

# Baseline eval on full DermNet test (no extra adaptation).
# Folder order quirk: DermNet ImageFolder gives acne=0, non_acne=1.
# The patch classifier was trained with folders ['negative','positive'], so **acne is class 1** there.
# Keep that mismatch in mind when mapping predictions back for metrics.
dermnet_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dermnet_test = ImageFolder('/content/dermnet_binary/test', transform=dermnet_transforms)
dermnet_loader = DataLoader(dermnet_test, batch_size=64, shuffle=False)
print("Classes (index order):", list(enumerate(dermnet_test.classes)))

model_cls.load_state_dict(torch.load('classifier_acne.pth'))
model_cls.eval()

all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for imgs, labels in dermnet_loader:
        imgs = imgs.to(device)
        out = model_cls(imgs)
        # In model space, acne is class 1
        prob_acne = torch.softmax(out, dim=1)[:, 1]
        preds_model = out.argmax(dim=1)
        # Remap to DermNet's labels (acne = 0)
        preds_dermnet = torch.where(
            preds_model == 1,
            torch.zeros_like(preds_model),
            torch.ones_like(preds_model),
        )
        all_preds.extend(preds_dermnet.cpu().numpy())
        all_probs.extend(prob_acne.cpu().numpy())
        all_labels.extend(labels.numpy())

# For F1/AUROC we treat acne as positive; on disk that's class 0 in DermNet.
acne_true = (np.array(all_labels) == 0).astype(np.int32)
acne_pred = (np.array(all_preds) == 0).astype(np.int32)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(acne_true, acne_pred)
auroc = roc_auc_score(acne_true, all_probs)

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}  (positive = acne)")
print(f"AUROC:    {auroc:.4f}  (score = P(acne), positive = acne)")

# Sanity check: folder order on disk
print("DermNet classes:", dermnet_loader.dataset.classes)
# You should see ['acne', 'non_acne'] → acne is index 0

# Patch dataset folder names the classifier saw
print("ACNE04 classes:", full_dataset.classes)
# Expect ['negative', 'positive'] -> acne (positive) = 1


acne: 312
non_acne: 3665
Classes (index order): [(0, 'acne'), (1, 'non_acne')]
Accuracy: 0.6978
F1 Score: 0.1900  (positive = acne)
AUROC:    0.6169  (score = P(acne), positive = acne)
DermNet classes: ['acne', 'non_acne']
ACNE04 classes: ['negative', 'positive']


## Apply domain adaptation techniques and document which ones you tried

**Prerequisites (run earlier cells first):** `/content/patches` built, ResNet baseline trained (`model_cls`, `full_dataset`, `train_loader`), `kaggle_path` set, DermNet binary test + `dermnet_loader` from the baseline eval cell.

This is a **long** code cell: histogram matching on patches, FaceNet `model_cls2`, second histogram pass + `model_cls3`, then 20-image pseudo-label fine-tune → `model_cls4`. Skipping patch or baseline training leaves weights stale or undefined.


In [12]:

# Histogram refs: random images from **all** DermNet train files in acne vs non-acne folders (large pools). The 20-image block at the end is **only** for pseudo-labeling, not for choosing histogram references.
def histogram_match(source, reference):
    source = np.array(source)
    reference = np.array(reference)
    matched = np.zeros_like(source)
    for c in range(3):
        src_hist, bins = np.histogram(source[:,:,c].flatten(), 256, [0,256])
        ref_hist, _ = np.histogram(reference[:,:,c].flatten(), 256, [0,256])
        src_cdf = src_hist.cumsum() / src_hist.sum()
        ref_cdf = ref_hist.cumsum() / ref_hist.sum()
        lookup = np.interp(src_cdf, ref_cdf, np.arange(256))
        matched[:,:,c] = lookup[source[:,:,c]]
    return Image.fromarray(matched.astype(np.uint8))

dermnet_train_dir = kaggle_path + '/train'
acne_folder_name = 'Acne and Rosacea Photos'

# 1. Collect all positive (Acne) image paths
acne_dir = os.path.join(dermnet_train_dir, acne_folder_name)
acne_image_paths = [os.path.join(acne_dir, f) for f in os.listdir(acne_dir)]

# 2. Collect all negative (Non-Acne) image paths from the OTHER 22 folders
non_acne_image_paths = []
for folder in os.listdir(dermnet_train_dir):
    if folder == acne_folder_name:
        continue  # Skip the acne folder!

    folder_path = os.path.join(dermnet_train_dir, folder)
    if os.path.isdir(folder_path):  # Make sure it's a folder
        for f in os.listdir(folder_path):
            non_acne_image_paths.append(os.path.join(folder_path, f))

print(f"Total Acne reference images available: {len(acne_image_paths)}")
print(f"Total Non-Acne reference images available: {len(non_acne_image_paths)}")

# Setup output directories
os.makedirs('/content/patches_matched/positive', exist_ok=True)
os.makedirs('/content/patches_matched/negative', exist_ok=True)

# 3. Match POSITIVE patches to random Acne references
print("Matching positive patches...")
for fname in os.listdir('/content/patches/positive'):
    src = Image.open(f'/content/patches/positive/{fname}').convert('RGB')

    # Pick a random path and open it
    random_ref_path = random.choice(acne_image_paths)
    ref_img = Image.open(random_ref_path).resize((64, 64))

    matched = histogram_match(src, ref_img)
    matched.save(f'/content/patches_matched/positive/{fname}')

# 4. Match NEGATIVE patches to random Non-Acne references
print("Matching negative patches...")
for fname in os.listdir('/content/patches/negative'):
    src = Image.open(f'/content/patches/negative/{fname}').convert('RGB')

    # Pick a random path from the giant list of 22 other diseases
    random_ref_path = random.choice(non_acne_image_paths)
    ref_img = Image.open(random_ref_path).resize((64, 64))

    matched = histogram_match(src, ref_img)
    matched.save(f'/content/patches_matched/negative/{fname}')

print("Done matching all patches!")

# FaceNet backbone pretrained on VGGFace2; classify=True gives logits
resnet = InceptionResnetV1(pretrained='vggface2', classify=True)

# This architecture exposes `.logits`, not `.fc`, for the last linear layer
resnet.logits = nn.Linear(resnet.logits.in_features, 2)

model_cls2 = resnet.to(device)

# Stronger aug + 160×160 for FaceNet
adapted_transforms = transforms.Compose([
    # FaceNet wants exactly 160×160
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Point the existing ImageFolder at the new transform
full_dataset.transform = adapted_transforms
train_set.dataset.transform = adapted_transforms

class_weights = torch.tensor([1.0, 5.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer2 = torch.optim.Adam(model_cls2.parameters(), lr=1e-4)

for epoch in range(10):
    model_cls2.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer2.zero_grad()
        out = model_cls2(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer2.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

torch.save(model_cls2.state_dict(), 'classifier_adapted.pth')

# --- Histogram matching: align patch colors to random DermNet refs ---
dermnet_train_dir = kaggle_path + '/train'
acne_folder_name = 'Acne and Rosacea Photos'

# Reference pool: acne folder
acne_dir = os.path.join(dermnet_train_dir, acne_folder_name)
acne_image_paths = [os.path.join(acne_dir, f) for f in os.listdir(acne_dir)]

# Reference pool: every non-acne condition
non_acne_image_paths = []
for folder in os.listdir(dermnet_train_dir):
    if folder == acne_folder_name:
        continue
    folder_path = os.path.join(dermnet_train_dir, folder)
    if os.path.isdir(folder_path):
        for f in os.listdir(folder_path):
            non_acne_image_paths.append(os.path.join(folder_path, f))

os.makedirs('/content/patches_matched/positive', exist_ok=True)
os.makedirs('/content/patches_matched/negative', exist_ok=True)

print("Matching positive patches to random Acne references...")
for fname in os.listdir('/content/patches/positive'):
    src = Image.open(f'/content/patches/positive/{fname}').convert('RGB')
    ref_img = Image.open(random.choice(acne_image_paths)).resize((64, 64))
    matched = histogram_match(src, ref_img)
    matched.save(f'/content/patches_matched/positive/{fname}')

print("Matching negative patches to random Non-Acne references...")
for fname in os.listdir('/content/patches/negative'):
    src = Image.open(f'/content/patches/negative/{fname}').convert('RGB')
    ref_img = Image.open(random.choice(non_acne_image_paths)).resize((64, 64))
    matched = histogram_match(src, ref_img)
    matched.save(f'/content/patches_matched/negative/{fname}')

print("Done matching patches!")

# --- Train on histogram-matched patches ---
# Reuse the 160×160 FaceNet transforms from above
matched_dataset = ImageFolder('/content/patches_matched', transform=adapted_transforms)

m_train_size = int(0.8 * len(matched_dataset))
m_val_size = len(matched_dataset) - m_train_size
m_train_set, m_val_set = random_split(matched_dataset, [m_train_size, m_val_size])

m_train_loader = DataLoader(m_train_set, batch_size=64, shuffle=True, num_workers=2)

# Fresh FaceNet head for matched patches
resnet3 = InceptionResnetV1(pretrained='vggface2', classify=True)
resnet3.logits = nn.Linear(resnet3.logits.in_features, 2)
model_cls3 = resnet3.to(device)

optimizer3 = torch.optim.Adam(model_cls3.parameters(), lr=1e-4)

# Train model_cls3
for epoch in range(10):
    model_cls3.train()
    total_loss = 0
    for imgs, labels in m_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer3.zero_grad()
        out = model_cls3(imgs)

        # Same class-weighted loss as cls2
        loss = criterion(out, labels)

        loss.backward()
        optimizer3.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

torch.save(model_cls3.state_dict(), 'classifier_histmatch.pth')

print(os.listdir(kaggle_path))

print(os.listdir(kaggle_path + '/train'))

# --- Grab 20 train images for pseudo-labeling (ignore folder names as supervision) ---
# Mix acne and other conditions so the pseudo-set isn't all one class.

random.seed(42)

UNLABELED_DIR = '/content/dermnet_unlabeled'
os.makedirs(UNLABELED_DIR, exist_ok=True)

train_root = os.path.join(kaggle_path, 'train')
acne_folder = os.path.join(train_root, 'Acne and Rosacea Photos')

# 10 acne-looking + 10 from random other folders
acne_imgs = glob.glob(os.path.join(acne_folder, '*.jpg'))
random.shuffle(acne_imgs)
picked = acne_imgs[:10]

non_acne_categories = [
    d for d in os.listdir(train_root)
    if os.path.isdir(os.path.join(train_root, d)) and d != 'Acne and Rosacea Photos'
]
non_acne_imgs = []
for cat in non_acne_categories:
    non_acne_imgs.extend(glob.glob(os.path.join(train_root, cat, '*.jpg')))
random.shuffle(non_acne_imgs)
picked += non_acne_imgs[:10]

# Flat folder: filenames only, no class subdirs
for src in picked:
    shutil.copy(src, os.path.join(UNLABELED_DIR, os.path.basename(src)))

print(f"Staged {len(os.listdir(UNLABELED_DIR))} unlabeled samples in {UNLABELED_DIR}")

# --- Pseudo-label with model_cls3, keep confident preds, fine-tune a copy ---
UNLABELED_DIR = '/content/dermnet_unlabeled'
unlabeled_paths = sorted(
    glob.glob(os.path.join(UNLABELED_DIR, '**', '*.*'), recursive=True)
)
unlabeled_paths = [p for p in unlabeled_paths
                   if p.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(unlabeled_paths)} unlabeled images")

# FaceNet-sized tensors for scoring
pl_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# Score each image; bucket by predicted class
model_cls3.eval()
by_class = defaultdict(list)  # {pred_class: [(path, pred, conf), ...]}

with torch.no_grad():
    for p in unlabeled_paths:
        img = Image.open(p).convert('RGB')
        x = pl_transform(img).unsqueeze(0).to(device)
        out = model_cls3(x)
        probs = torch.softmax(out, dim=1)[0]
        conf, pred = probs.max(0)
        by_class[int(pred.item())].append(
            (p, int(pred.item()), float(conf.item()))
        )

# Take top-K confident per predicted class so classes stay balanced
K = 5
pseudo_data = []
for cls, items in by_class.items():
    items.sort(key=lambda t: -t[2])
    pseudo_data.extend(items[:K])

print(f"Balanced pseudo-labels: {len(pseudo_data)} total "
      f"({len(by_class.get(1, []))} acne predicted, "
      f"{len(by_class.get(0, []))} non_acne predicted)")
for p, y, c in pseudo_data:
    print(f"  {'acne' if y == 1 else 'non_acne'}  (conf={c:.2f})  {os.path.basename(p)}")

assert len(pseudo_data) > 0, "No pseudo-labels generated."

# Small Dataset for the kept pseudo-labeled paths
class PseudoLabeledDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label, _ = self.items[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

# Mild aug — only ~10 images may survive filtering
pl_train_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
])
pseudo_ds = PseudoLabeledDataset(pseudo_data, pl_train_transform)
pseudo_loader = DataLoader(pseudo_ds, batch_size=8, shuffle=True)

# Fine-tune a deep copy of cls3 with a small LR so you don't erase source training.
model_cls4 = copy.deepcopy(model_cls3)

optimizer_pl = torch.optim.Adam(model_cls4.parameters(), lr=1e-5)
criterion_pl = nn.CrossEntropyLoss()

EPOCHS_PL = 5
for epoch in range(EPOCHS_PL):
    model_cls4.train()
    total = 0.0
    for imgs, labels in pseudo_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_pl.zero_grad()
        out = model_cls4(imgs)
        loss = criterion_pl(out, labels)
        loss.backward()
        optimizer_pl.step()
        total += loss.item()
    print(f"[Pseudo-label FT] Epoch {epoch+1} | Loss: {total:.4f}")

torch.save(model_cls4.state_dict(), 'classifier_pseudolabel.pth')


Total Acne reference images available: 840
Total Non-Acne reference images available: 14717
Matching positive patches...
Matching negative patches...
Done matching all patches!


  0%|          | 0.00/107M [00:00<?, ?B/s]

Epoch 1 | Loss: 148.8346
Epoch 2 | Loss: 111.4781
Epoch 3 | Loss: 103.1041
Epoch 4 | Loss: 97.1207
Epoch 5 | Loss: 102.6934
Epoch 6 | Loss: 88.7774
Epoch 7 | Loss: 88.1812
Epoch 8 | Loss: 83.5120
Epoch 9 | Loss: 85.3754
Epoch 10 | Loss: 79.4185
Matching positive patches to random Acne references...
Matching negative patches to random Non-Acne references...
Done matching patches!
Epoch 1 | Loss: 256.1684
Epoch 2 | Loss: 188.6736
Epoch 3 | Loss: 166.7688
Epoch 4 | Loss: 152.9119
Epoch 5 | Loss: 148.6040
Epoch 6 | Loss: 145.1289
Epoch 7 | Loss: 134.1811
Epoch 8 | Loss: 137.0292
Epoch 9 | Loss: 128.9189
Epoch 10 | Loss: 122.6149
['train', 'test']
['Bullous Disease Photos', 'Urticaria Hives', 'Exanthems and Drug Eruptions', 'Vascular Tumors', 'Systemic Disease', 'Warts Molluscum and other Viral Infections', 'Eczema Photos', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Herpes HPV and other STDs Photos', 'Melanoma Skin Cancer Nevi and Moles', 'Actinic Keratosis Basal Cell Carcinoma a

## Evaluate on full DermNet test set: Accuracy, F1, AUROC

Run baseline first, then the summary cell that evaluates `model_cls`, `model_cls2`, `model_cls3`, and `model_cls4` on the same `dermnet_loader`. Refresh the Reflection section to match your latest printed DataFrame.


In [13]:

def evaluate_on_dermnet(model, img_size, threshold=None):
    """Accuracy, F1, and AUROC on `dermnet_loader`.

    Label convention: patch training uses acne = **class 1** (`positive` folder after sorting); DermNet `ImageFolder` uses acne = **class 0** (`acne` before `non_acne`). We use `prob_acne = softmax[:, 1]` and treat DermNet label 0 as acne-positive for F1/AUROC.

    If threshold is None, picks the F1-best threshold on P(acne) from this same run
    (fine for reporting; use a val split if you need a rigorously tuned threshold)."""
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in dermnet_loader:
            imgs = imgs.to(device)
            imgs = F.interpolate(imgs, size=img_size, mode='bilinear', align_corners=False)
            out = model(imgs)
            # Training used acne = class 1
            prob_acne = torch.softmax(out, dim=1)[:, 1]
            all_probs.extend(prob_acne.cpu().numpy())
            all_labels.extend(labels.numpy())

    arr_probs  = np.array(all_probs)
    arr_labels = np.array(all_labels)

    # DermNet stores acne as class 0 — that's our positive for these metrics
    acne_true = (arr_labels == 0).astype(int)

    # If no threshold passed, maximize F1 on this sweep
    if threshold is None:
        prec, rec, thr = precision_recall_curve(acne_true, arr_probs)
        f1s = 2 * prec * rec / (prec + rec + 1e-9)
        best_idx = np.nanargmax(f1s[:-1])  # PR curve: last point has no paired threshold
        threshold = thr[best_idx]

    acne_pred = (arr_probs >= threshold).astype(int)
    # Accuracy wants DermNet's original 0/1 labels
    arr_preds_dermnet = np.where(acne_pred == 1, 0, 1)

    return {
        "Accuracy":  accuracy_score(arr_labels, arr_preds_dermnet),
        "F1":        f1_score(acne_true, acne_pred),
        "AUROC":     roc_auc_score(acne_true, arr_probs),
        "Threshold": float(threshold),
    }


# Each backbone expects a different spatial size
r1 = evaluate_on_dermnet(model_cls,  img_size=(64, 64))    # Baseline ResNet18
r2 = evaluate_on_dermnet(model_cls2, img_size=(160, 160))  # FaceNet + Augmentation
r3 = evaluate_on_dermnet(model_cls3, img_size=(160, 160))  # FaceNet + Aug + Histogram Match
r4 = evaluate_on_dermnet(model_cls4, img_size=(160, 160))  # + Pseudo-labeling

df = pd.DataFrame(
    [r1, r2, r3, r4],
    index=["Baseline", "Augmentation", "Aug + Histogram Match", "Aug + HM + Pseudo-label"],
)
print(df.round(4))


                         Accuracy      F1   AUROC  Threshold
Baseline                   0.6970  0.1918  0.6169     0.4960
Augmentation               0.7714  0.1759  0.5883     0.9252
Aug + Histogram Match      0.6213  0.1938  0.6340     0.1296
Aug + HM + Pseudo-label    0.6123  0.1901  0.6174     0.3105


## Grad-CAM on DermNet (10+ predictions)

Assignment requirement: visualize model attention on **at least 10** DermNet test images. Below we run Grad-CAM for three adapted checkpoints: augmentation-only (`model_cls2`), augmentation + histogram match (`model_cls3`), and augmentation + histogram match + pseudo-labeling (`model_cls4`).

## Grad-CAM: augmentation-only (`model_cls2`)

_Intermediate adaptation (saved weights `classifier_adapted.pth`). Figure: `gradcam_dermnet_cls2.png`._

## Grad-CAM: augmentation + histogram match (`model_cls3`)

_Histogram-matched adaptation checkpoint. Figure: `gradcam_dermnet_cls3.png`._

## Grad-CAM: augmentation + histogram match + pseudo-labeling (`model_cls4`)

_Pseudo-label fine-tuned checkpoint. Figure: `gradcam_dermnet_cls4.png`._


In [14]:
%matplotlib inline

# Grad-CAM targets FaceNet `block8` only — run after `model_cls2`/`3`/`4` exist in memory.

def generate_gradcam_visualizations(model, model_title, save_name):
    model.eval()

    # Grad-CAM target: FaceNet's last conv block
    target_layer = [model.block8]
    cam = GradCAM(model=model, target_layers=target_layer)

    # FaceNet expects 160×160
    dermnet_raw = ImageFolder('/content/dermnet_binary/test', transform=transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor()
    ]))

    # 5 acne + 5 non-acne so the grid isn't one-sided
    by_label = defaultdict(list)
    for i in range(len(dermnet_raw)):
        by_label[dermnet_raw[i][1]].append(i)
    indices_to_show = by_label[0][:5] + by_label[1][:5]

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()

    shown = 0
    for i in indices_to_show:
        img_tensor, label = dermnet_raw[i]
        img_np = img_tensor.permute(1, 2, 0).numpy()

        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        input_tensor = norm(img_tensor).unsqueeze(0).to(device)

        # Explain logits for class 1 (acne) in model space
        grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(1)])[0]
        visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        with torch.no_grad():
            out = model(input_tensor)
            pred = out.argmax(1).item()
            prob_acne = torch.softmax(out, dim=1)[0, 1].item()

        # pred uses model indices; true uses DermNet folder names
        pred_label = 'acne' if pred == 1 else 'non_acne'
        true_label = dermnet_raw.classes[label]
        correct = pred_label == true_label

        axes[shown].imshow(visualization)
        axes[shown].set_title(
            f'True: {true_label}\nPred: {pred_label} ({prob_acne:.2f})',
            fontsize=10,
            color='green' if correct else 'red'
        )
        axes[shown].axis('off')
        shown += 1
    plt.suptitle(model_title, fontsize=16)
    plt.tight_layout()
    plt.savefig(save_name, dpi=150)
    plt.show()

# Generate the three figures used in the report
generate_gradcam_visualizations(
    model_cls2,
    model_title='Grad-CAM: FaceNet + Augmentation (model_cls2)',
    save_name='gradcam_dermnet_cls2.png'
)

generate_gradcam_visualizations(
    model_cls3,
    model_title='Grad-CAM: FaceNet + Aug + Histogram Match (model_cls3)',
    save_name='gradcam_dermnet_cls3.png'
)

generate_gradcam_visualizations(
    model_cls4,
    model_title='Grad-CAM: FaceNet + Aug + HM + Pseudo-label (model_cls4)',
    save_name='gradcam_dermnet_cls4.png'
)



Output hidden; open in https://colab.research.google.com to view.

## Reflection: Domain Transfer

Cross-domain transfer from ACNE04 patches to full DermNet images is difficult: patches are tight 64×64 crops around lesions, while DermNet photos vary in lighting, framing, and condition mix.

**DermNet test results (acne = positive class):**

| Model | Accuracy | F1 | AUROC | Threshold |
|-------|----------|-----|-------|-----------|
| Baseline (ResNet-18) | 0.6970 | 0.1918 | 0.6169 | 0.4960 |
| FaceNet + Augmentation | 0.7714 | 0.1759 | 0.5883 | 0.9252 |
| Aug + Histogram Match | 0.6213 | 0.1938 | 0.6340 | 0.1296 |
| Aug + HM + Pseudo-label | 0.6123 | 0.1901 | 0.6174 | 0.3105 |

**The headline finding:** AUROC remains in a narrow band (~0.59–0.63). Histogram matching is best in this run (0.6340), but the margin over baseline (0.6169) is small enough that I treat it as suggestive rather than conclusive at single-seed.

**Why accuracy can look better while AUROC gets worse:** Augmentation-only has the highest accuracy (0.7714) but the lowest AUROC (0.5883), and its F1-optimal threshold is very high (0.9252). That pattern indicates a conservative operating point that favors the non-acne majority class, not stronger acne discrimination.

**What Grad-CAM reveals (the real story):** Across the 30 panels (5 acne + 5 non-acne × 3 checkpoints), high-activation regions repeatedly overlap or sit near the DermNet watermark/text area. The shortcut persists across cls2/cls3/cls4, which aligns with the weak AUROC separation among adaptation methods.

**Caveats on rigor:** All numbers are single-seed runs. Re-running shifted the rank/order across some checkpoints, so I would not claim method superiority without 3–5 seeds and confidence intervals.

**If I had more time:** (1) Address the shortcut directly — inpaint/crop watermark regions or apply artifact-robust augmentation. (2) Report PR-AUC alongside AUROC for the 1:22 imbalance. (3) Add balanced sampling or focal loss. (4) Try learned alignment methods (DANN/CORAL). (5) Run multi-seed experiments before claiming small deltas.
